# 01 - Data Exploration

Explore the datasets we'll use for testing LLM document parsing capabilities.

**Datasets:**
- FUNSD: Noisy scanned forms
- SROIE: Degraded receipts
- NOD: Documents at multiple noise levels

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from src.data_loader import DataLoader, list_available_datasets

In [ ]:
# Check what's available
available = list_available_datasets("../data")
print("Dataset availability:")
for name, ready in available.items():
    status = "✓ Ready" if ready else "✗ Not downloaded"
    print(f"  {name}: {status}")

In [ ]:
loader = DataLoader("../data")

## FUNSD - Noisy Scanned Forms

199 real scanned forms with annotations for:
- Questions (field labels)
- Answers (field values)
- Headers
- Other text

In [ ]:
# Load FUNSD
try:
    funsd_docs = list(loader.load_funsd(split="test"))
    print(f"Loaded {len(funsd_docs)} FUNSD test documents")
except FileNotFoundError as e:
    print(f"FUNSD not available: {e}")
    print("Run: python src/download_data.py")
    funsd_docs = []

In [ ]:
# Visualize sample FUNSD documents
if funsd_docs:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, doc in enumerate(funsd_docs[:6]):
        img = doc.load_image()
        axes[i].imshow(img)
        axes[i].set_title(f"{doc.id}")
        axes[i].axis('off')
    
    plt.suptitle("FUNSD Sample Documents", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Examine ground truth structure
if funsd_docs:
    sample = funsd_docs[0]
    print(f"Document: {sample.id}")
    print(f"\nGround truth keys: {sample.ground_truth.keys()}")
    print(f"\nHeaders: {sample.ground_truth.get('header', [])[:3]}")
    print(f"\nQuestions: {sample.ground_truth.get('question', [])[:5]}")
    print(f"\nAnswers: {sample.ground_truth.get('answer', [])[:5]}")
    print(f"\nKey-Value Pairs: {sample.ground_truth.get('key_value_pairs', [])[:3]}")

## SROIE - Degraded Receipts

1000 receipt images with ground truth for:
- Company name
- Address
- Date
- Total amount

In [ ]:
# Load SROIE
try:
    sroie_docs = list(loader.load_sroie(split="test"))
    print(f"Loaded {len(sroie_docs)} SROIE test documents")
except FileNotFoundError as e:
    print(f"SROIE not available: {e}")
    sroie_docs = []

In [ ]:
# Visualize sample SROIE documents
if sroie_docs:
    fig, axes = plt.subplots(2, 3, figsize=(12, 16))
    axes = axes.flatten()
    
    for i, doc in enumerate(sroie_docs[:6]):
        img = doc.load_image()
        axes[i].imshow(img)
        axes[i].set_title(f"{doc.id}")
        axes[i].axis('off')
    
    plt.suptitle("SROIE Sample Receipts", fontsize=14)
    plt.tight_layout()
    plt.show()

## Sample Documents

Quick test with synthetic or custom documents.

In [ ]:
# Load samples
sample_docs = list(loader.load_samples())
print(f"Found {len(sample_docs)} sample documents")

for doc in sample_docs:
    print(f"  - {doc.id}: {doc.image_path}")

In [ ]:
# Display samples
if sample_docs:
    for doc in sample_docs[:4]:
        img = doc.load_image()
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(doc.id)
        plt.axis('off')
        plt.show()

## Dataset Statistics

In [ ]:
# Compile statistics
stats = []

if funsd_docs:
    stats.append({
        "Dataset": "FUNSD",
        "Documents": len(funsd_docs),
        "Has Ground Truth": sum(1 for d in funsd_docs if d.ground_truth),
        "Type": "Scanned forms"
    })

if sroie_docs:
    stats.append({
        "Dataset": "SROIE",
        "Documents": len(sroie_docs),
        "Has Ground Truth": sum(1 for d in sroie_docs if d.ground_truth),
        "Type": "Receipts"
    })

if sample_docs:
    stats.append({
        "Dataset": "Samples",
        "Documents": len(sample_docs),
        "Has Ground Truth": 0,
        "Type": "Test documents"
    })

if stats:
    df = pd.DataFrame(stats)
    display(df)
else:
    print("No datasets loaded. Run: python src/download_data.py")

## Next Steps

1. Download missing datasets: `python src/download_data.py`
2. Run extraction experiments: `02_extraction_experiments.ipynb`
3. Test chaos tolerance: `03_chaos_gradient_test.ipynb`